In [2]:
import os
import pandas as pd
import psycopg2
import warnings

In [3]:
warnings.filterwarnings('ignore')
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [4]:
def read_url(link):
    """ Creates a pandas DataFrame from data online
    - Parameters:
        - link: link to the zipped data
    - Returns:
    """
    import io
    import requests
    import pandas as pd

    # Define URL and extract information
    response = requests.get(link)
    content = response.content
    # Convert into a Pandas DataFrame
    df = pd.read_csv(io.BytesIO(content), sep=',', compression='gzip')

    return df

In [5]:
listings = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/listings.csv.gz')
#listings = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2026-03-30/data/listings.csv.gz')

In [7]:
listings['instant_bookable'].head()

0    f
1    f
2    f
3    f
4    f
Name: instant_bookable, dtype: object

In [434]:
column_dates = ['last_scraped', 'host_since', 'price_quote_checkin_date',
                'price_quote_checkout_date', 'calendar_updated', 'calendar_last_scraped',
                'first_review', 'last_review']

for col in column_dates:
    if col in listings.columns.to_list():
        listings[col] = pd.to_datetime(listings[col], errors='coerce')
        listings[col] = listings[col].dt.date
        listings[col] = listings[col].where(listings[col].notna(), None)
    
listings = listings.astype(object)
listings = listings.where(pd.notnull(listings), None)

In [ ]:
if 'price' in listings.columns.to_list():
    listings['price'] = listings['price'].replace('[\$,]', '', regex=True).astype(float)
if 'host_acceptance_rate' in listings.columns.to_list():
    listings['host_acceptance_rate'] = listings['host_acceptance_rate'].replace('[%,]', '', regex=True).astype(float)
if 'host_response_rate' in listings.columns.to_list():
    listings['host_response_rate'] = listings['host_response_rate'].replace('[%,]', '', regex=True).astype(float)
if 'price_quote_total_price' in listings.columns.to_list():
    listings['price_quote_total_price'] = listings['price_quote_total_price'].replace('[\$,]', '', regex=True).astype(float)
if 'price_quote_price_per_night' in listings.columns.to_list():
    listings['price_quote_price_per_night'] = listings['price_quote_price_per_night'].replace('[\$,]', '', regex=True).astype(float)


In [436]:
# Create connection to the database and initialize it
def create_db_connection() -> psycopg2.extensions.connection:
    conn = psycopg2.connect(
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5433"),
        dbname=os.getenv("DB_NAME", "smartbnb"),
        user=os.getenv("DB_USER", "admin"),
        password=os.getenv("DB_PASSWORD", "admin")
    )
    return conn

def drop_connection(conn):
    conn.close()

In [437]:
conn = create_db_connection()
columns = listings.columns.to_list()
columns_names = ", ".join(columns)
placeholders = ", ".join(["%s"] * len(columns))

update_clause = ", ".join(
    [
        f"{col} = COALESCE(EXCLUDED.{col}, listings.{col})"
        for col in columns
        if col != "id"
    ]
)

query = f"""
    INSERT INTO listings ({columns_names}) 
    VALUES ({placeholders})
    ON CONFLICT (id) DO UPDATE SET {update_clause}
    """
    # ON CONFLICT (id) DO NOTHING

with conn.cursor() as cur:
    for _, row in listings.iterrows():
        values = [
            None if pd.isna(value) else value 
            for value in row
        ]
        cur.execute(query, values)

conn.commit()
#drop_connection()

In [438]:
print(
    listings["host_since"].apply(
        lambda x: type(x).__name__
    ).value_counts()
)

host_since
date        25487
NoneType      914
Name: count, dtype: int64
